# **Section 1: Revision Part 3 - Tables**

Revision notes for Creating tables, selecting, filtering, sorting, chaining, and the difference between a table and an
array. Assumes `section_1_revision_python` and `section_1_revision_arrays`.

**This notebook reads real files.** Keep `cones.csv`, `wcplayerstatistics2026.csv` and
`du_bois.csv` in the same folder as the notebook, or the reading cells will fail.

In [ ]:
# Setup - run this first
%pip install -q datascience ipywidgets

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np

---

## **Contents**

1. [Two ways to create a table](#1)
2. [Choosing columns, choosing rows](#2)
3. [A real data set, and the trap in it](#3)
4. [`.select()` returns a table, `.column()` returns an array](#4)
5. [Chains read left to right](#5)
6. [Real data brings type surprises](#6)
7. [W.E.B. Du Bois was a data scientist](#7)
8. [**>>Quick questions<<**](#8)
9. [**>>Self-check<<**](#9)
10. [Quick reference](#10)


---

<a id='1'></a>
## **1. Two ways to create a table**

**a) From a file:**

```
Table.read_table('filename.csv')
table.show(n)
```

The filename is a string, so it goes in quotes, and the file must sit in the same folder as the
notebook. `.show(n)` displays the first *n* rows and `.show()` displays all of them.

In [ ]:
# Always look at a few rows before doing anything else
cones = Table.read_table('cones.csv')
cones.show(3)

**b) From scratch:**

```
Table().with_columns(label, values)
Table().with_columns(label1, values1, label2, values2, ...)
```

`Table()` is an empty table, `label` is a string, and `values` is an array. Every column must be the
same length. Both forms return a **new** table and leave the original unchanged.

In [ ]:
# Built from scratch: a string label and an array of values, twice
Table().with_columns(
    'Street', make_array('Jorissen', 'Yale', 'Jan Smuts', 'Enoch Sontonga'),
    'Blocks from campus', np.arange(4)
)

---

<a id='2'></a>
## **2. Choosing columns, choosing rows**

| Form | Keeps | Returns |
|---|---|---|
| `table.select(label, ...)` | only the columns named | a table |
| `table.drop(label, ...)` | everything except those named | a table |
| `table.where(label, value)` | only the rows that match | a table |
| `table.sort(label)` | every row, reordered | a table |

`select` and `drop` work on **columns**, `where` works on **rows**, and `sort` reorders them.
All four return a new table, so the original keeps everything unless you reassign it.

Column labels are strings and always go in quotes. `cones.select(Flavor)` without quotes is a
`NameError`, because Python looks for a name rather than a label.

`sort` is ascending by default, and `descending=True` is a named argument, exactly like `ndigits`
on `round`.

In [ ]:
# Two columns, cheapest first
cones.select('Flavor', 'Price').sort('Price')

In [ ]:
# Rows, not columns: every column comes back, only chocolate rows stay
cones.where('Flavor', 'chocolate')

**Watch out.** If nothing matches, `where` gives you an **empty table**, not an error. Python has
done exactly what you asked; the question was just wrong. That silence is the subject of the next
section.

---

<a id='3'></a>
## **3. A real data set, and the trap in it**

`wcplayerstatistics2026.csv` holds one row per player at the 2026 World Cup, 1039 rows and 26
columns. Positions are stored as clean two-letter codes, so filtering on them behaves as you would
expect.

In [ ]:
# 236 forwards out of 1039 players
wc = Table.read_table('wcplayerstatistics2026.csv')
wc.where('Pos', 'FW').num_rows

Now try the same thing on `Squad`. Filtering for Egypt the obvious way returns an empty table and
no error at all.

In [ ]:
# Empty, silently: 'Egypt' is not what the file stores
wc.where('Squad', 'Egypt')

```
table.column(column_label)
table.column(column_label).item(index)
```

`.column` returns a column's values as an **array**, and chaining `.item(0)` shows you a single one.
This is how you find out what a column actually contains, rather than what you assume it contains.

In [ ]:
# The stored values carry a country-code prefix
wc.column('Squad').item(0)

In [ ]:
# With the prefix, the filter finds all 21 players
egypt = wc.where('Squad', 'eg Egypt')
egypt.num_rows

`Club` is stored the same way, but worse: the prefix carries a league and a division, as in
`1.eng Leeds United`. The format varies between leagues, so it cannot be guessed. Inspect the
column.

The habit worth building from this: **look at one real value before you filter on a column.** One
`.item(0)` costs a second and saves you from an empty table you cannot explain.

---

<a id='4'></a>
## **4. `.select()` returns a table, `.column()` returns an array**

| Form | Returns | Use it when |
|---|---|---|
| `table.select(label)` | a **table** with one column | you will keep working with it as a table |
| `table.column(label)` | an **array** of values | you will do arithmetic on it |

Same data, two containers. Tell them apart by how they display: a table has a heading and a border,
an array has square brackets and commas. Array functions such as `np.average` need `.column`, and
they fail on the table that `.select` gives back.

This is the point where the arrays notebook pays off. A column of a table is an array, so
everything you know about `sum`, `len`, `.item` and element-wise arithmetic applies to it.

In [ ]:
# .column gives an array, which np.average can work with
np.average(egypt.column('Min'))

In [ ]:
# Sort, take the column as an array, take the first element
egypt.sort('Min', descending=True).column('Player').item(0)

---

<a id='5'></a>
## **5. Chains read left to right**

```
table.method1(...).method2(...)
```

Each method runs on whatever the previous one produced, so at every step ask what kind of thing you
are holding and whether it still has what the next step needs.

Order therefore matters. `wc.drop('Pos').where('Pos', 'FW')` fails, because the column is gone
before the filter looks for it. Filter first, drop afterwards.

In [ ]:
# Filter first, then drop the column you filtered on
wc.where('Pos', 'FW').drop('Pos').select('Player', 'Squad', 'Gls').show(3)

One more chaining rule worth memorising: `.show()` returns nothing at all. It prints the table and
hands back `None`, so any method chained after it raises an `AttributeError`. Put `.show()` last, or
leave it off and let Jupyter display the table itself.

---

<a id='6'></a>
## **6. Real data brings type surprises**

Ages, birth years and match counts in this file all arrive with a `.0` on the end. They are floats,
not ints, because that is how the file stores them. Nothing has gone wrong, but it will surprise you
the first time you try to build a label out of one.

In [ ]:
# Age reads as 25.0, and its type confirms why
wc.column('Age').item(0), type(wc.column('Age').item(0))

In [ ]:
# int on a float truncates; the decimal part here is zero
int(wc.column('Born').item(0))

Going via `str` instead fails: `str(2000.0)` is the text `'2000.0'`, and `int` cannot parse a
string containing a decimal point. For a numeric string, go via float, as in the Python notebook.

---

<a id='7'></a>
## **7. W.E.B. Du Bois was a data scientist**

`du_bois.csv` holds the 1900 household expenditure data Du Bois presented at the Paris Exposition:
one row per income class, with the proportion of income spent on rent, food, clothes, taxes and
everything else.

Two things to notice. `num_rows`, `num_columns` and `labels` are **attributes**, not methods, so
they take no brackets: you are asking the table for something it already knows rather than asking it
to compute something.

In [ ]:
# Attributes take no brackets
du_bois = Table.read_table('du_bois.csv')
du_bois.num_rows, du_bois.num_columns

In [ ]:
# The highest-earning class, found by sorting rather than by guessing a label
du_bois.sort('ACTUAL AVERAGE', descending=True).take(0)

Sorting is safer than filtering here, because it does not require you to know in advance what the
highest bracket is called. Run `du_bois.labels` first whenever you are unsure of a column name.

**Try it yourself.** The `FOOD` column is a proportion and `ACTUAL AVERAGE` is an amount, so
multiplying them element-wise gives the money each class spent on food. Add that to the table as a
new column.

In [ ]:
# Your turn: multiply the two columns as arrays, then add the result with with_column

---

<a id='8'></a>
## **8. Quick questions**

Five of them. Set `my_answer` to a letter and run the cell.
A wrong answer gets a nudge so you can try again; a right one gets the reason.

These come before the written questions below on purpose: they are quicker,
and they check the things people most often get wrong.

In [ ]:
# Run this once. mcq.py must be in the same folder as this notebook.
from mcq import check_answer, show_answer

**M1.** Which one can you pass straight to `np.average`?

**a)** `tbl.select('Min')`  
**b)** `tbl.column('Min')`  
**c)** either of them  
**d)** neither, you need `np.mean` for a table  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w1t_m1', my_answer)

**M2.** `wc.where('Squad', 'Egypt')` runs and gives you an empty table. What has happened?

**a)** there are no Egyptian players in the data  
**b)** `where` needs `are.equal_to`  
**c)** the stored values are not spelled the way you assumed  
**d)** the column name is wrong  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w1t_m2', my_answer)

**M3.** One of these fails. Which, and why?

**a)** `wc.drop('Pos').where('Pos', 'FW')`, because the column has gone  
**b)** `wc.where('Pos', 'FW').drop('Pos')`, because you cannot drop after filtering  
**c)** neither fails  
**d)** both fail  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w1t_m3', my_answer)

**M4.** Why does `du_bois.num_rows()` fail when `du_bois.num_rows` works?

**a)** the table is empty  
**b)** you need to pass it a column name  
**c)** `num_rows` only works after `sort`  
**d)** `num_rows` is an attribute, not a method  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w1t_m4', my_answer)

**M5.** You run `cones.drop('Price')` and `cones` still shows Price. Is something broken?

**a)** yes, `drop` has failed  
**b)** no, `drop` returns a new table and leaves the original alone  
**c)** no, but only because `Price` is the last column  
**d)** yes, you need `drop_column`  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w1t_m5', my_answer)

---

<a id='9'></a>
## **9. Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** What type does `wc.select('Min')` return? What type does `wc.column('Min')` return?

<details>
<summary><strong>Answer</strong></summary>

<code>.select</code> returns a <strong>Table</strong> with one labelled column. <code>.column</code> returns an <strong>array</strong> of values with no label. Use <code>.column</code> before <code>np.average</code>, and <code>.select</code> when the next step is another table method.

</details>

**Q2.** What error does `cones.show(3).sort('Price')` raise, and why?

<details>
<summary><strong>Answer</strong></summary>

<code>AttributeError: 'NoneType' object has no attribute 'sort'</code>. <code>.show()</code> prints the table and returns <code>None</code>, so there is no table left for <code>.sort</code> to run on. Put <code>.show()</code> last in a chain.

</details>

**Q3.** `wc.where('Squad', 'Egypt')` runs without an error but gives you nothing. What has happened?

<details>
<summary><strong>Answer</strong></summary>

The filter worked correctly and found no matching rows, because the stored values carry a country-code prefix and read <code>'eg Egypt'</code>. An empty table is an answer, not an error. Check a real value with <code>wc.column('Squad').item(0)</code> before filtering.

</details>

**Q4.** Which of `wc.drop('Pos').where('Pos', 'FW')` and `wc.where('Pos', 'FW').drop('Pos')` fails, and why?

<details>
<summary><strong>Answer</strong></summary>

The <strong>first</strong> fails. Chains run left to right, so the drop removes <code>Pos</code> before <code>where</code> looks for it. The second filters while the column is still there and drops it afterwards, which is what you want.

</details>

**Q5.** Why does `du_bois.num_rows()` fail when `du_bois.num_rows` works?

<details>
<summary><strong>Answer</strong></summary>

<code>num_rows</code> is an <strong>attribute</strong>, a value the table already holds, not a method to be called. Adding brackets tries to call an integer, which raises a <code>TypeError</code>. The same applies to <code>num_columns</code> and <code>labels</code>.

</details>

**Q6.** `Born` holds values like `2000.0`. Which of `int(wc.column('Born').item(0))` and `int(str(wc.column('Born').item(0)))` gives the year, and which errors?

<details>
<summary><strong>Answer</strong></summary>

The first gives <strong>2000</strong>: <code>int</code> truncates a float, and the decimal part here is zero. The second raises a <code>ValueError</code>, because <code>str</code> produces <code>'2000.0'</code> and <code>int</code> cannot parse a string with a decimal point in it.

</details>

**Q7.** You want the name of the Egyptian player with the most minutes, not the whole row. Write the chain.

<details>
<summary><strong>Answer</strong></summary>

<code>egypt.sort('Min', descending=True).column('Player').item(0)</code>. Sort descending, take the column as an array, then take position 0. Using <code>.select('Player')</code> instead would leave you with a one-column table that you cannot index with <code>.item</code>.

</details>

**Q8.** You run `cones.drop('Price')` and then `cones` still shows the Price column. Is something broken?

<details>
<summary><strong>Answer</strong></summary>

No. Every table method returns a <strong>new</strong> table and leaves the original alone. If you want to keep the result, give it a name: <code>cones_without_price = cones.drop('Price')</code>.

</details>

---

<a id='10'></a>
## **10. Cheatsheet**

| Form | What it does | Returns |
|---|---|---|
| `Table.read_table('f.csv')` | reads a table from a file | table |
| `Table().with_columns(l, v, ...)` | builds a table from labels and arrays | table |
| `table.show(3)` | prints the first 3 rows | nothing (`None`) |
| `table.select('A', 'B')` | keeps only those columns | table |
| `table.drop('A')` | keeps everything else | table |
| `table.where('A', value)` | keeps matching rows; empty if none match | table |
| `table.sort('A')` | reorders rows, ascending | table |
| `table.sort('A', descending=True)` | reorders rows, descending | table |
| `table.column('A')` | one column's values | array |
| `table.take(0)` | the row at a position | table |
| `table.num_rows` | how many rows (no brackets) | number |
| `table.num_columns` | how many columns (no brackets) | number |
| `table.labels` | the column labels (no brackets) | labels |

**Reminder:** look at one real value before filtering on a column, and check whether you
are holding a table or an array before you do arithmetic on it.